In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
import pingouin as pg

import patsy

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')


In [2]:
# 读取数据
df = pd.read_excel(r'E:\pycharm all files\眼动数据处理\完整眼动数据分析\眼动数据格式对齐后文件.xlsx')
df

,阶段,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE),组别,性别,受试者,飞行天数
0,起飞,13.0,1.398246,0.507883,Alcohol,女,付瑞晗,1
1,转弯,4.0,0.890165,0.212143,Alcohol,女,付瑞晗,1
2,转弯,10.0,0.923579,0.399419,Alcohol,女,付瑞晗,1
3,巡航,18.0,1.012136,0.642191,Alcohol,女,付瑞晗,1
4,转弯,12.0,1.664170,0.542276,Alcohol,女,付瑞晗,1
...,...,...,...,...,...,...,...,...
1450,转弯,38.0,1.856486,0.533725,Control,男,郭浚杰,7
1451,巡航,38.0,2.609610,0.506751,Control,男,郭浚杰,7
1452,转弯,31.0,2.074730,0.374989,Control,男,郭浚杰,7
1453,转弯,0.0,0.654472,0.000000,Control,男,郭浚杰,7


In [7]:
df_heatmap_data = df[df["阶段"] == "降落"]
df_heatmap_data = df_heatmap_data.drop(columns=["阶段"])
df_heatmap_data.to_excel("eye_热力图降落阶段数据.xlsx")

In [3]:
# 1. 对 AOI转换次数 拟合混合效应模型
# -----------------------------
# 固定效应：组别、飞行天数、阶段
# 随机效应：受试者
# 混合效应模型（AOI转换次数）
model_AOI = smf.mixedlm(
    "AOI转换次数 ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_AOI = model_AOI.fit(method="powell", maxiter=500)
print(result_AOI.summary())

# 2. 对 静态注释熵(SGE) 拟合混合效应模型
# -----------------------------
# 固定效应：组别、飞行天数、阶段
# 随机效应：受试者
# 混合效应模型（静态注释熵(SGE)）
model_SGE = smf.mixedlm(
    "Q('静态注释熵(SGE)') ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_SGE = model_SGE.fit(method="powell", maxiter=500)
print(result_SGE.summary())

# 3. 对 眼跳注视熵(GTE) 拟合混合效应模型
# -----------------------------
# 固定效应：组别、飞行天数、阶段
# 随机效应：受试者
# 混合效应模型（眼跳注视熵(GTE)）
model_GTE = smf.mixedlm(
    "Q('眼跳注视熵(GTE)') ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_GTE = model_GTE.fit(method="powell", maxiter=500)
print(result_GTE.summary())

                 Mixed Linear Model Regression Results
Model:                 MixedLM      Dependent Variable:      AOI转换次数   
No. Observations:      1455         Method:                  REML      
No. Groups:            32           Scale:                   89.7008   
Min. group size:       25           Log-Likelihood:          -5336.6171
Max. group size:       49           Converged:               Yes       
Mean group size:       45.5                                            
-----------------------------------------------------------------------
                            Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
-----------------------------------------------------------------------
Intercept                   12.448    2.166  5.747 0.000   8.203 16.693
组别[T.Control]               -5.537    3.044 -1.819 0.069 -11.503  0.429
阶段[T.起飞]                    -5.567    2.995 -1.859 0.063 -11.437  0.304
阶段[T.转弯]                    -5.253    2.372 -2.215 0.027  -9.901 -0.604
阶段[T.降落] 

In [4]:
# 结果保存到excel
# 提取结果为 DataFrame
def extract_result(result, model_name):
    summary_df = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "z_value": result.tvalues,
        "p_value": result.pvalues
    })
    summary_df["显著性"] = summary_df["p_value"].apply(lambda p: "显著" if p < 0.05 else "不显著")
    return summary_df


# 保留模型结果
df_AOI = extract_result(result_AOI, "AOI转换次数")
df_SGE = extract_result(result_SGE, "SGE均值")
df_GTE = extract_result(result_GTE, "GTE均值")

# 合并三个模型结果
df_all = pd.concat([df_AOI, df_SGE, df_GTE], axis=0)

# 提取显著结果
df_sig = df_all[df_all["显著性"] == "显著"].copy()

# 保存到 Excel
with pd.ExcelWriter("混合效应模型结果.xlsx") as writer:
    # 各模型单独的结果
    df_AOI.to_excel(writer, sheet_name="AOI转换次数", index=True)
    df_SGE.to_excel(writer, sheet_name="静态注视熵(SGE)", index=True)
    df_GTE.to_excel(writer, sheet_name="眼跳注视熵(GTE)", index=True)

    # 所有结果合并
    df_all.to_excel(writer, sheet_name="全部结果汇总", index=True)

    # 仅显著结果
    df_sig.to_excel(writer, sheet_name="显著结果汇总", index=True)
    print('保存成功！')

保存成功！
